In [ ]:
from transformers import BertTokenizer, BertModel
import torch
import numpy as np
from sklearn.cluster import KMeans
from nltk.tokenize import sent_tokenize


import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
def split_into_sentences(text):
    return sent_tokenize(text)

text = """Экстрактивная суммаризация на основе вхождения общих слов

Данный алгоритм очень прост как для понимания, так и дальнейшей реализации. Здесь мы работаем только с исходным текстом, и по большому счету у нас отсутствует потребность в обучении какой-либо модели извлечения. В моем случае извлекаемые информационные блоки будут представлять собой определенные предложения текста.

Итак, на первом шаге разбиваем входной текст на предложения и каждое предложение разбиваем на токены (отдельные слова), проводим для них лемматизацию (приведение слова к «канонической» форме). Этот шаг необходим для того, чтобы алгоритм объединял одинаковые по смыслу слова, но отличающиеся по словоформам.

Затем задаем функцию схожести для каждой пары предложений. Она будет рассчитываться как отношение числа общих слов, встречающихся в обоих предложениях, к их суммарной длине. В результате мы получим коэффициенты схожести для каждой пары предложений.

Предварительно отсеяв предложения, которые не имеют общих слов с другими, построим граф, где вершинами являются сами предложения, ребра между которыми показывают наличие в них общих слов."""
sentences = split_into_sentences(text)
print(sentences)

['Экстрактивная суммаризация на основе вхождения общих слов\n\nДанный алгоритм очень прост как для понимания, так и дальнейшей реализации.', 'Здесь мы работаем только с исходным текстом, и по большому счету у нас отсутствует потребность в обучении какой-либо модели извлечения.', 'В моем случае извлекаемые информационные блоки будут представлять собой определенные предложения текста.', 'Итак, на первом шаге разбиваем входной текст на предложения и каждое предложение разбиваем на токены (отдельные слова), проводим для них лемматизацию (приведение слова к «канонической» форме).', 'Этот шаг необходим для того, чтобы алгоритм объединял одинаковые по смыслу слова, но отличающиеся по словоформам.', 'Затем задаем функцию схожести для каждой пары предложений.', 'Она будет рассчитываться как отношение числа общих слов, встречающихся в обоих предложениях, к их суммарной длине.', 'В результате мы получим коэффициенты схожести для каждой пары предложений.', 'Предварительно отсеяв предложения, котор

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

def encode_sentences(sentences):
    inputs = tokenizer(sentences, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    sentence_embeddings = outputs.last_hidden_state.mean(dim=1)
    return sentence_embeddings

sentence_embeddings = encode_sentences(sentences)

In [ ]:
def cluster_sentences(sentence_embeddings, num_clusters):
    kmeans = KMeans(n_clusters=num_clusters)
    kmeans.fit(sentence_embeddings)
    return kmeans.labels_

num_clusters = 2  # Количество кластеров == количество предложений в результате
labels = cluster_sentences(sentence_embeddings, num_clusters)

/usr/local/lib/python3.10/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [ ]:
def extract_summary(sentences, labels, num_clusters):
    summary = []
    for i in range(num_clusters):
        cluster_sentences = [sentences[j] for j in range(len(labels)) if labels[j] == i]
        summary.append(cluster_sentences[0])  # Включаем первое предложение из каждого кластера
    return ' '.join(summary)

summary = extract_summary(sentences, labels, num_clusters)
print("Summary:")
print(summary)

Summary:
Итак, на первом шаге разбиваем входной текст на предложения и каждое предложение разбиваем на токены (отдельные слова), проводим для них лемматизацию (приведение слова к «канонической» форме). Экстрактивная суммаризация на основе вхождения общих слов

Данный алгоритм очень прост как для понимания, так и дальнейшей реализации.
